# CS 297 - Experiment 1: Tiny Recursive Model
### Description: Supervised image augmentation severity, noise detection, and routing
### Author: Ethan Machado

Nine native-resolution RGB patches → shared lightweight CNN → official TRM reasoning blocks → severity 0–3 → existing noise/severity/route JSON.

This is a supervised counterpart to the Qwen experiment. It uses a person-disjoint 35/5/10 split, jointly trains the encoder and TRM, and tests only the best validation checkpoint. Patch sampling and supervised learning differ from Qwen's whole-image zero-shot inference. No denoiser runs here. See `plans/` and `PIPELINE.md` for the architecture and validation record.

## Setup and configuration
Select the project `.venv` kernel. Install `requirements-trm.txt` once using the documented shell command. Restart and Run All creates a new training session and held-out pass.

Set `TRAIN_MODEL=False` and `CHECKPOINT_PATH` below to evaluate a saved checkpoint using its original split/configuration. The `TRM_*` environment overrides support small synthetic smoke runs; leave them unset for the real dataset.

In [ ]:
import os
import sys
import json
from pathlib import Path
from datetime import datetime, timezone
from uuid import uuid4

PROJECT_PATH = Path.cwd().resolve()
if not (PROJECT_PATH / "src/image_quality_pipeline.py").exists():
    PROJECT_PATH = PROJECT_PATH.parent
if not (PROJECT_PATH / "src/image_quality_pipeline.py").exists():
    raise RuntimeError("Start from the project root or notebooks directory")
if str(PROJECT_PATH) not in sys.path:
    sys.path.insert(0, str(PROJECT_PATH))

import torch
import pandas as pd
from IPython.display import display
from src.image_quality_pipeline import build_dataset
from src.trm_data import (create_subject_split, load_subject_split, partition_frame,
                          load_image_patches, validate_subject_split, validate_image_sizes)
from src.trm_model import VisionTRM, UPSTREAM_SHA
from src.trm_training import (train_experiment, load_checkpoint, select_device,
                              environment_info, seed_everything)
from src.trm_adapter import TRMImageModel, evaluate_checkpoint, EXPERIMENT1_PROMPT

TRAIN_MODEL = os.environ.get("TRM_TRAIN_MODEL", "true").lower() == "true"
CHECKPOINT_PATH = Path(os.environ["TRM_CHECKPOINT_PATH"]) if os.environ.get("TRM_CHECKPOINT_PATH") else None
CONFIG_PATH = Path(os.environ.get("TRM_CONFIG_PATH", str(PROJECT_PATH / "configs/trm_experiment1.json")))
DATA_PROJECT_ROOT = Path(os.environ.get("TRM_DATA_PROJECT_ROOT", str(PROJECT_PATH))).resolve()
OUTPUT_ROOT = Path(os.environ.get("TRM_OUTPUT_ROOT", str(PROJECT_PATH / "outputs/trm_supervised_runs")))
SESSION_ID = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ") + "_" + uuid4().hex[:8]
SESSION_DIR = OUTPUT_ROOT / SESSION_ID
RUN_DIR = SESSION_DIR / "experiment_001/run_001"
config = json.loads(CONFIG_PATH.read_text())
if not TRAIN_MODEL:
    if CHECKPOINT_PATH is None:
        raise ValueError("Set CHECKPOINT_PATH for checkpoint-only evaluation")
    _, original_checkpoint = load_checkpoint(CHECKPOINT_PATH, device="cpu")
    config = original_checkpoint["config"]
seed_everything(config["seed"])
device = select_device(config["training"]["device"])
print("Session:", SESSION_DIR)
print("Pinned upstream:", UPSTREAM_SHA)
display(environment_info(device))
display(config)
ORIGINAL_PROMPT = EXPERIMENT1_PROMPT  # metadata only; TRM does not consume text


## Dataset validation, person split, and patch preview
All four severity variants of each person stay in one partition. The persisted manifest stores relative paths, labels, and content hashes. It is verified on checkpoint-only evaluation. File paths and person IDs do not enter the model.

In [ ]:
df_experiment = build_dataset(DATA_PROJECT_ROOT / "data/files")
validate_image_sizes(df_experiment, config["preprocessing"])
if TRAIN_MODEL:
    split = create_subject_split(df_experiment, config["split"]["seed"],
                                 project_root=DATA_PROJECT_ROOT, counts=config["split"]["counts"])
else:
    split = validate_subject_split(original_checkpoint["split"], df_experiment, DATA_PROJECT_ROOT)
frames = {name: partition_frame(split, name, DATA_PROJECT_ROOT) for name in ("train", "validation", "test")}
display(pd.DataFrame({name: frame.groupby("severity").size() for name, frame in frames.items()}))
preview = load_image_patches(frames["train"].file_path.iloc[0], config["preprocessing"])
print("Patch tensor:", tuple(preview.shape), preview.dtype)
from PIL import Image
patch_size = config["preprocessing"]["patch_size"]
grid_size = config["preprocessing"]["grid_size"]
canvas = Image.new("RGB", (grid_size * patch_size, grid_size * patch_size))
for index, patch in enumerate(preview):
    pixels = (patch.permute(1, 2, 0).numpy() * 255).round().astype("uint8")
    canvas.paste(Image.fromarray(pixels), ((index % grid_size) * patch_size, (index // grid_size) * patch_size))
display(canvas)


## Encoder and recursive model
The shared two-layer reasoner uses noncausal attention, rotary positions, residual RMS normalization, and SwiGLU. Answer/working states reset per batch. Each refinement performs `H_cycles × (L_cycles + 1)` reasoner calls. Earlier H cycles have no gradients; the last has gradients. States detach between outer refinements. No token/puzzle embeddings or adaptive-halting heads are instantiated.

In [ ]:
model_preview = VisionTRM(config["model"], patch_count=config["preprocessing"]["grid_size"] ** 2)
display(model_preview.parameter_counts())
print("Reasoner calls per image:", config["model"]["refinement_steps"] * config["model"]["H_cycles"] * (config["model"]["L_cycles"] + 1))
del model_preview


## Train and select checkpoint
Each minibatch gets one optimizer update per outer refinement. The image encoder is recomputed with current weights at each step. Select the earliest checkpoint with validation cross-entropy improving by more than `1e-6`; early stopping uses only validation. The test partition is reserved for the final saved pass. Interrupted/failed training retains its best completed checkpoint and records that status.

In [ ]:
if TRAIN_MODEL:
    CHECKPOINT_PATH = train_experiment(df_experiment, DATA_PROJECT_ROOT, SESSION_DIR, config)
    training_manifest = json.loads((SESSION_DIR / "training_manifest.json").read_text())
    display(training_manifest)
    display(pd.read_csv(SESSION_DIR / "training_history.csv"))
else:
    print("Checkpoint-only evaluation:", CHECKPOINT_PATH)


## Reload the best model
The checkpoint reconstructs the encoder, recursion, preprocessing, and original split without external model downloads.

In [ ]:
adapter = TRMImageModel.from_checkpoint(CHECKPOINT_PATH, device=str(device))
display(adapter.inference_config)


## Held-out Experiment 1 inference
Evaluate only the test partition through the shared runner, batch size one, with supervised mode and the original prompt retained as metadata. It writes the existing manifest, labels, per-image batch JSON, and metrics. Explicit run folders are never overwritten.

In [ ]:
RUN_DIR = evaluate_checkpoint(CHECKPOINT_PATH, df_experiment, DATA_PROJECT_ROOT, SESSION_DIR,
                              device=str(device), prompt=ORIGINAL_PROMPT)
print("Saved held-out run:", RUN_DIR)


## Offline review and CSV export
This section can run independently: set `RUN_DIR` to a saved `experiment_001/run_001` folder and run the next cell. It requires saved JSON and the shared evaluator; it does not load a checkpoint, TRM, Qwen, or a GPU. Confusion matrices use true rows/predicted columns. Failures and unattempted images remain visible.

In [ ]:
from pathlib import Path
import sys
import pandas as pd
from IPython.display import display
review_root = Path.cwd().resolve()
if not (review_root / "src/evaluation.py").exists():
    review_root = review_root.parent
if str(review_root) not in sys.path:
    sys.path.insert(0, str(review_root))
from src.trm_reporting import export_saved_run
if "RUN_DIR" not in globals():
    raise RuntimeError("Set RUN_DIR to the saved run folder before offline evaluation")
report = export_saved_run(RUN_DIR)
print("FULL RUN COMPLETE" if report["complete"] else "INCOMPLETE RUN — partial results only")
print("Training status:", report["training_status"])
print("Planned:", report["planned"], "Attempted:", report["attempted"],
      "Unattempted:", report["planned"] - report["attempted"])
print("Inference seconds:", report["inference_seconds"])
display(report["summary"])
for task, values in report["metrics"].items():
    print(task, "confusion matrix (valid predictions)")
    display(pd.DataFrame(values["confusion_matrix"],
                         index=pd.Index(values["labels"], name="True"),
                         columns=pd.Index(values["labels"], name="Predicted")))
    print(task, "per-class metrics")
    display(pd.DataFrame(values["per_class"]).T)
display(report["coverage"])
display(report["predictions"].head(20))
print("Failures:", len(report["failures"]))
display(report["failures"])
print("JSON, metrics, and four CSV exports:", RUN_DIR)
